In [ ]:
def main(datasources, start_date, end_date):
    """
    Haar小波多尺度成交脉冲持续性因子。

    将分钟成交量分解到2、4、8、16...分钟尺度，
    测量2-8分钟能量占比、尺度集中度及四个日内区段的一致性。
    输出越高，表示短尺度成交脉冲越集中且全天更持续。
    """
    import numpy as np
    import pandas as pd
    import dai

    table_name = datasources["bar1m"]
    eps = 1e-12
    min_points = 100

    sql = f"""
    SELECT
        date::DATETIME AS minute_time,
        date::DATE::DATETIME AS date,
        instrument::STRING AS instrument,
        CAST(volume AS DOUBLE) AS cumulative_volume
    FROM {table_name}
    WHERE date >= '{start_date}'
      AND date <= '{end_date}'
      AND volume IS NOT NULL
      AND volume >= 0
      AND NOT (
          EXTRACT(hour FROM date) = 9
          AND EXTRACT(minute FROM date) = 30
      )
      AND (
          EXTRACT(hour FROM date) < 14
          OR (
              EXTRACT(hour FROM date) = 14
              AND EXTRACT(minute FROM date) <= 57
          )
      )
    ORDER BY instrument, date, minute_time
    """

    data = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    pool = dai.query(
        """
        SELECT date::DATE::DATETIME AS date,
               instrument::STRING AS instrument
        FROM bigalpha_2026_instruments
        """,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    pool["date"] = pd.to_datetime(pool["date"], errors="coerce").dt.normalize()
    pool["instrument"] = pool["instrument"].astype(str)
    pool = pool.dropna().drop_duplicates(["date", "instrument"])

    if data.empty:
        pool["factor"] = 0.0
        return pool[["date", "instrument", "factor"]]

    data["minute_time"] = pd.to_datetime(data["minute_time"], errors="coerce")
    data["date"] = pd.to_datetime(data["date"], errors="coerce").dt.normalize()
    data["instrument"] = data["instrument"].astype(str)
    data["cumulative_volume"] = pd.to_numeric(
        data["cumulative_volume"], errors="coerce"
    )
    data = data.dropna().drop_duplicates(
        ["minute_time", "instrument"], keep="last"
    ).sort_values(["instrument", "date", "minute_time"])
    keys = ["date", "instrument"]
    data["minute_volume"] = (
        data.groupby(keys)["cumulative_volume"].diff().clip(lower=0)
    )

    def prepare(values):
        values = np.asarray(values, dtype="float64")
        values = values[np.isfinite(values)]
        if values.size < 16:
            return values
        q25, q75 = np.quantile(values, [0.25, 0.75])
        iqr = q75 - q25
        upper = q75 + 3.0 * iqr if iqr > eps else np.quantile(values, 0.99)
        values = np.clip(values, 0.0, max(float(upper), 0.0))
        values = values - np.mean(values)
        scale = np.std(values)
        return values / scale if scale > eps else np.zeros_like(values)

    def haar_energy(values):
        approximation = prepare(values)
        if approximation.size < 16:
            return np.array([], dtype="float64")
        target_length = 1 << int(np.ceil(np.log2(approximation.size)))
        if target_length > approximation.size:
            approximation = np.pad(
                approximation,
                (0, target_length - approximation.size),
                mode="reflect",
            )
        energies = []
        while approximation.size >= 4:
            even = approximation[0::2]
            odd = approximation[1::2]
            detail = (even - odd) / np.sqrt(2.0)
            approximation = (even + odd) / np.sqrt(2.0)
            energies.append(float(np.sum(detail**2)))
        return np.asarray(energies, dtype="float64")

    def wavelet_score(values):
        energies = haar_energy(values)
        if energies.size < 3 or np.sum(energies) <= eps:
            return np.nan
        probabilities = energies / (np.sum(energies) + eps)
        target_share = float(np.sum(probabilities[:3]))
        entropy = -np.sum(
            probabilities * np.log(probabilities + eps)
        ) / np.log(max(probabilities.size, 2))
        return float(target_share * max(1.0 - entropy, 0.0))

    def calculate_factor(group):
        values = group["minute_volume"].dropna().to_numpy()
        if values.size < min_points:
            return np.nan
        whole = wavelet_score(values)
        if not np.isfinite(whole):
            return np.nan
        block_scores = []
        for block in np.array_split(values, 4):
            score = wavelet_score(block)
            if np.isfinite(score):
                block_scores.append(score)
        if len(block_scores) >= 3:
            block_scores = np.asarray(block_scores)
            persistence = 1.0 - np.std(block_scores) / (
                np.mean(np.abs(block_scores)) + eps
            )
            persistence = float(np.clip(persistence, 0.0, 1.0))
        else:
            persistence = 0.0
        return float(whole * (0.5 + 0.5 * persistence))

    factor_data = (
        data.groupby(keys, sort=False)
        .apply(calculate_factor)
        .rename("factor")
        .reset_index()
    )
    result = pool.merge(factor_data, how="left", on=keys)
    result["factor"] = pd.to_numeric(
        result["factor"], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)
    result["factor"] = (
        result.groupby("date")["factor"]
        .transform(lambda x: x.fillna(x.median()))
        .fillna(0.0)
    )
    return (
        result[["date", "instrument", "factor"]]
        .drop_duplicates(keys, keep="last")
        .sort_values(keys)
        .reset_index(drop=True)
    )


if __name__ == "__main__":
    from bigmodule import M
    import dai

    datasources = {"bar1m": "bigalpha_2026_stock_bar1m"}
    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"
    factor_data = main(datasources, start_date, end_date)
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]}, compression=True,
    ).df()
    print(M.bigalpha_eval._latest(
        factor_data=factor_data, factor_pool=factor_pool,
        process_pools=False, show=True, start_date=start_date, end_date=end_date,
    ))